# Implement Attention from Scratch
### Problem Statement
Multi-Head Attention (MHA) is the bread-and-butter of the Transformer architecture. It enables the model to **jointly attend** to information from different representation subspaces at different positions.

Your goal is to implement MHA from scratch using PyTorch, simulating exactly what `torch.nn.MultiheadAttention` does — projecting Q, K, V for each head, computing attention weights, applying them to V, and concatenating the outputs across all heads.

---

### Requirements

1. **Linear Projections for Q, K, V**
   - Project input `q`, `k`, `v` into a total of `d_model` dimensions.
   - Split them into `num_heads` of `d_head = d_model // num_heads` each.

2. **Scaled Dot-Product Attention per Head**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply an optional `mask` before softmax.
   - Use the scores to weight `V`.

3. **Combine the Heads**
   - Concatenate the outputs of all heads.
   - Apply a final linear projection to restore the shape: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch’s Reference**
   - Test your output against `torch.nn.MultiheadAttention` using the same input tensors.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Make sure all tensors are reshaped properly when splitting and combining heads.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when heads and shape are aligned.

---

<details>
  <summary>💡 Hint</summary>

  - Use `.view()` and `.transpose()` to shape Q, K, V to `(batch_size, num_heads, seq_len, d_head)`.
  - Softmax should be applied over the **last dimension** (attention scores across sequence).
  - Use `.contiguous().view()` to flatten the multi-head outputs back into `(batch_size, seq_len, d_model)`.
  - Match PyTorch’s behavior using the same projections and batch-first format.

</details>

---

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [4]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

torch.Size([3, 4, 8])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    Implements multi-head attention.
    """
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads  # Head size dimension

        # TODO: Create linear projections for Q, K, V and output
        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)


    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: Multi-head attention output of shape (batch_size, seq_len, d_model)
        """
        # TODO: Implement forward pass
        # 1. Project Q, K, V
        # 2. Reshape to (batch_size, num_heads, seq_len, d_head)
        # 3. Compute scaled dot-product attention
        # 4. Apply mask if provided
        # 5. Apply softmax
        # 6. Weight values with attention weights
        # 7. Combine heads and apply output projection
        batch_size, seq_len, _ = q.shape
        Q = self.Q_w(q)  # (batch_size, seq_len, d_model)
        K = self.K_w(k)
        V = self.V_w(v)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2) # batch_size,num_heads,seq_len,d_head

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5) # batch size, num_heads, seq_len, seq_len
        attn_weights = F.softmax(scores, dim=-1) # batch size, num_heads, seq_len, seq_len

        output = torch.matmul(attn_weights, V) # (batch_size, num_heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_out(output)
        

In [6]:
# Testing on data & compare
# Create PyTorch's MultiheadAttention
multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True)

# Create our custom implementation
custom_attn = MultiHeadAttention(num_heads, d_model)

# Copy weights from PyTorch's implementation to ours for fair comparison
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

# Now compare outputs
output_custom = custom_attn(q, k, v)
print("Custom output:")
print(output_custom)

output, _ = multihead_attn(q, k, v)
print("\nPyTorch output:")
print(output)

assert torch.allclose(output_custom, output, atol=1e-06, rtol=1e-05)  # Check if they are close enough.
print("\n✅ Outputs match! Implementation is correct.")

Custom output:
tensor([[[ 7.1699e-02,  5.3411e-02,  3.3034e-01,  1.5892e-01,  6.5924e-02,
           2.3432e-01,  1.2263e-01,  2.1005e-01],
         [ 7.5689e-02,  5.2544e-02,  3.3219e-01,  1.5638e-01,  6.8600e-02,
           2.3832e-01,  1.2223e-01,  2.1144e-01],
         [ 7.2534e-02,  5.3125e-02,  3.3081e-01,  1.5864e-01,  6.6714e-02,
           2.3542e-01,  1.2263e-01,  2.1010e-01],
         [ 6.9490e-02,  5.1792e-02,  3.3143e-01,  1.6153e-01,  6.6810e-02,
           2.3345e-01,  1.2220e-01,  2.0941e-01]],

        [[-4.5193e-02,  2.6360e-02,  2.6671e-01,  2.7284e-01,  9.3438e-02,
           1.5845e-01,  1.2320e-01,  1.5793e-01],
         [-4.3427e-02,  2.6275e-02,  2.6286e-01,  2.7077e-01,  9.2779e-02,
           1.5760e-01,  1.2343e-01,  1.5535e-01],
         [-4.5840e-02,  2.6753e-02,  2.6641e-01,  2.7329e-01,  9.4146e-02,
           1.5864e-01,  1.2388e-01,  1.5618e-01],
         [-4.1810e-02,  2.5378e-02,  2.6161e-01,  2.6910e-01,  9.1674e-02,
           1.5723e-01,  1.2250e-0